# Chapter 7 Lab — Megatron Parallelism Config → Effective GPUs & Batch

**Estimated time:** 25 minutes · **Read-and-run.** Compute world size and global batch from
a parallelism config, and reason about within-node vs across-node placement.

In [ ]:
cfg = dict(
  tensor_parallel_size=4,     # TP: splits each layer's matmuls across GPUs (chatty -> within a node)
  pipeline_parallel_size=2,   # PP: splits layers into stages
  data_parallel_size=8,       # DP: replicas over different data
  micro_batch_size=2,
  grad_accum_steps=8,
  gpus_per_node=8)
for k,v in cfg.items(): print(f'{k:22s}= {v}')

In [ ]:
world = cfg['tensor_parallel_size']*cfg['pipeline_parallel_size']*cfg['data_parallel_size']
nodes = -(-world // cfg['gpus_per_node'])  # ceil div
global_batch = cfg['micro_batch_size']*cfg['data_parallel_size']*cfg['grad_accum_steps']
print(f'world size (total GPUs)  = {world}')
print(f'nodes (@{cfg["gpus_per_node"]}/node)      = {nodes}')
print(f'global batch size       = {global_batch}')

tp=cfg['tensor_parallel_size']; gpn=cfg['gpus_per_node']
print('\nPlacement check:')
print(f'  TP group size {tp} <= gpus_per_node {gpn}? {"YES - keep TP within a node (NVLink)" if tp<=gpn else "NO - TP spans nodes, slow!"}')

## Your turn
Set `tensor_parallel_size=16` with `gpus_per_node=8` and re-run. Why is that placement a
performance trap? (Hint: TP is the most bandwidth-hungry dimension.)

## Key takeaway
Parallelism choice follows the bottleneck: **TP** for layers too big for one GPU (keep it on
NVLink within a node), **PP** to fit depth across nodes, **DP** to scale throughput. Interconnect
speed dictates which dimension goes within vs across nodes.